# Evaluación 2 — Ingeniería y Visualización de Datos de Transporte

**Autor:** Felipe Ignacio Zapata Lagos · Ingeniería en Informática · INACAP
**Asignatura:** Visualización de Datos · **Unidad:** Diseño e implementación de gráficos para datos

**Fuente:** Junta de Aeronáutica Civil (JAC), *Operaciones de Aeronaves*, Portal de Datos Abiertos
(https://datos.gob.cl/dataset/operaciones-aeronaves), licencia CC0. Detalle de la investigación de
la fuente (Parte I) y de la formulación del problema (Parte II) en `docs/`.

**Pregunta principal.** ¿Cómo se distribuye y cómo ha evolucionado la actividad aeroportuaria
registrada en Chile entre 2000 y 2026 según aeropuerto, tipo de tráfico (doméstico/internacional),
tamaño de aeronave y momento del día, y qué patrones temporales la caracterizan?

| | Pregunta secundaria |
|---|---|
| **P1** | ¿Qué aeropuertos concentran las operaciones y qué peso tiene el tráfico internacional en cada uno? |
| **P2** | ¿Cómo evolucionaron las operaciones domésticas e internacionales entre 2000 y 2025 y cómo se recuperó cada una tras la caída de 2020? |
| **P3** | ¿Cómo se distribuyen las operaciones según tamaño de aeronave y hora del día, y difieren esos patrones entre segmentos? |

**H1 (exploratoria, no es conclusión):** tras la caída de 2020, el tráfico internacional se
recuperó más lentamente que el doméstico respecto de su nivel de 2019.

> Regla de trabajo: **dato → evidencia → interpretación → posible acción**. No se afirma
> causalidad a partir de patrones conjuntos.

## 1. Importaciones

In [1]:
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import seaborn as sns

## 2. Configuración

Las rutas se resuelven desde la raíz del proyecto para que el notebook funcione igual si se
ejecuta desde `notebooks/` o desde la raíz.

In [2]:
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))

import estilo  # noqa: E402
import preparacion as prep  # noqa: E402

RAW = RAIZ / "data" / "raw"
PROCESSED = RAIZ / "data" / "processed"
FIGURAS = RAIZ / "output" / "figures"
for carpeta in (PROCESSED, FIGURAS):
    carpeta.mkdir(parents=True, exist_ok=True)

RUTA_CSV = RAW / "operaciones-aeropuertos.csv"
RUTA_PARQUET = RAW / "bitacora-vuelos.parquet"
if not RUTA_PARQUET.exists():
    raise FileNotFoundError(
        "Falta data/raw/bitacora-vuelos.parquet (~138 MB, no se versiona en GitHub). "
        "Ejecute desde la raíz:  python src/descargar_datos.py"
    )

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
estilo.aplicar_estilo()
print("Raíz del proyecto:", RAIZ)
print("pandas", pd.__version__, "| seaborn", sns.__version__)

Raíz del proyecto: /home/claude/evaluacion_02_transportes
pandas 3.0.5 | seaborn 0.13.2


## 3. Carga de datos

Se utilizan **ambos recursos** del conjunto:

* `operaciones-aeropuertos.csv` — resumen mensual por aeropuerto y tipo de tráfico (sin sobrevuelos).
* `bitacora-vuelos.parquet` — un registro por operación (aterrizaje, despegue o sobrevuelo).

El parquet se lee con `pyarrow` convirtiendo los textos a `category` y el `pmd` (decimal) a
`float64`; leerlo directamente con `pd.read_parquet` ocupa ~2,6 GB de RAM, lo que puede
hacer inviable el análisis en un computador personal.

In [3]:
csv_raw = pd.read_csv(RUTA_CSV)
t0 = time.perf_counter()
bit_raw = prep.cargar_bitacora(RUTA_PARQUET)
print(f"Bitácora cargada en {time.perf_counter() - t0:.1f} s | "
      f"{bit_raw.memory_usage(deep=True).sum() / 1e6:,.0f} MB en memoria")
print("CSV:", csv_raw.shape, "| Bitácora:", bit_raw.shape)

Bitácora cargada en 3.0 s | 359 MB en memoria
CSV: (16172, 4) | Bitácora: (11163791, 12)


## 4. Comparación de formatos (Parte VI)

**Los dos archivos publicados NO son equivalentes:** el CSV es un *resumen agregado*
(aeropuerto × mes × tipo de tráfico, 4 columnas) y el parquet es el *registro individual*
(11 millones de operaciones, 12 columnas). Comparar directamente su tiempo de carga o
su tamaño sería una comparación artificial: no contienen la misma información.

Por eso se hace (a) una descripción de cada archivo por separado y (b) un experimento
controlado: **el mismo subconjunto de datos** (bitácora 2025) se escribe en CSV y en Parquet
y se comparan tamaño, tiempo de carga, memoria y tipos inferidos.

In [4]:
meta = pq.ParquetFile(RUTA_PARQUET).metadata
print("Parquet publicado:", f"{RUTA_PARQUET.stat().st_size / 1e6:,.1f} MB,",
      f"{meta.num_rows:,} filas, {meta.num_row_groups} row groups, creado con {meta.created_by}")
print("CSV publicado:    ", f"{RUTA_CSV.stat().st_size / 1e6:,.2f} MB,", f"{len(csv_raw):,} filas")
print("\nTipos declarados en el esquema parquet (se conservan al leer):")
print(pq.read_schema(RUTA_PARQUET))
print("\nTipos inferidos por pandas al leer el CSV (no hay esquema; se infieren del texto):")
print(csv_raw.dtypes)

Parquet publicado: 138.2 MB, 11,163,791 filas, 91 row groups, creado con Polars
CSV publicado:     0.29 MB, 16,172 filas

Tipos declarados en el esquema parquet (se conservan al leer):
dt_operacion: timestamp[us, tz=America/Santiago]
aeropuerto_oaci: large_string
tipo_operacion: large_string
aeropuerto_dgac_orig_dest: large_string
aerolinea_dgac: large_string
numero_vuelo: large_string
actividad_cod: large_string
matricula: large_string
modelo_avion: large_string
modelo_avion_desc: large_string
es_internacional: bool
pmd: decimal128(10, 5)

Tipos inferidos por pandas al leer el CSV (no hay esquema; se infieren del texto):
mes_id                     int64
aeropuerto_oaci              str
internacional_domestico      str
cnt_operaciones            int64
dtype: object


In [5]:
experimento = RAIZ / "data" / "tmp_formatos"
experimento.mkdir(exist_ok=True)
sub = bit_raw[bit_raw["dt_operacion"].dt.year == 2025].copy()
sub["pmd"] = sub["pmd"].astype("float64")
for c in sub.select_dtypes("category"):
    sub[c] = sub[c].astype("string")
sub.to_csv(experimento / "bitacora_2025.csv", index=False)
sub.to_parquet(experimento / "bitacora_2025.parquet", index=False, compression="zstd")

filas = []
for fmt, lector in [("csv", pd.read_csv), ("parquet", pd.read_parquet)]:
    ruta = experimento / f"bitacora_2025.{fmt}"
    tiempos = []
    for _ in range(3):
        t0 = time.perf_counter()
        d = lector(ruta)
        tiempos.append(time.perf_counter() - t0)
    filas.append({
        "formato": fmt,
        "tamaño_MB": ruta.stat().st_size / 1e6,
        "carga_s (mediana de 3)": float(np.median(tiempos)),
        "memoria_MB": d.memory_usage(deep=True).sum() / 1e6,
        "dt_operacion": str(d["dt_operacion"].dtype),
        "es_internacional": str(d["es_internacional"].dtype),
        "pmd": str(d["pmd"].dtype),
    })
comparacion_formatos = pd.DataFrame(filas).set_index("formato")
comparacion_formatos

,tamaño_MB,carga_s (mediana de 3),memoria_MB,dt_operacion,es_internacional,pmd
formato,,,,,,
csv,47.675850,1.007138,82.927248,str,bool,float64
parquet,6.572796,0.083808,68.659948,"datetime64[us, America/Santiago]",bool,float64


**Lectura.** Con los mismos datos, Parquet ocupa una fracción del tamaño del CSV, se carga más
rápido y **conserva los tipos** (fecha con zona horaria y booleano), mientras que el CSV
entrega la fecha como texto y obliga a reconvertirla. Para la bitácora (11 M de filas) Parquet
es claramente el formato conveniente; para el resumen mensual (16 mil filas, 285 KB) el CSV es
suficiente y tiene la ventaja de abrirse en cualquier herramienta. *La carpeta temporal del
experimento no se versiona.*

In [6]:
import shutil  # noqa: E402

shutil.rmtree(experimento)
del sub, d

## 5. Perfilamiento (Partes VII y VIII)

### 5.1 Resumen mensual por aeropuerto (CSV)

In [7]:
csv_raw.head()

,mes_id,aeropuerto_oaci,internacional_domestico,cnt_operaciones
0,200001,SCAP,D,6
1,200001,SCAR,D,691
2,200001,SCAR,I,139
3,200001,SCBA,D,382
4,200001,SCCC,D,82


In [8]:
csv_raw.tail()

,mes_id,aeropuerto_oaci,internacional_domestico,cnt_operaciones
16167,202609,SCTE,D,1318
16168,202609,SCTN,D,412
16169,202609,SCVD,D,108
16170,202609,SCVM,D,798
16171,202609,SCVM,I,2


In [9]:
csv_raw.sample(5, random_state=42)

,mes_id,aeropuerto_oaci,internacional_domestico,cnt_operaciones
1071,200112,SCFA,I,7
11808,202002,SCEL,D,10249
13622,202302,SCAR,D,531
291,200007,SCDA,D,1360
9294,201604,SCFT,D,4


In [10]:
print("shape:", csv_raw.shape)
print("columnas:", list(csv_raw.columns))
csv_raw.info()

shape: (16172, 4)
columnas: ['mes_id', 'aeropuerto_oaci', 'internacional_domestico', 'cnt_operaciones']
<class 'pandas.DataFrame'>
RangeIndex: 16172 entries, 0 to 16171
Data columns (total 4 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   mes_id                   16172 non-null  int64
 1   aeropuerto_oaci          16172 non-null  str  
 2   internacional_domestico  16172 non-null  str  
 3   cnt_operaciones          16172 non-null  int64
dtypes: int64(2), str(2)
memory usage: 584.5 KB


In [11]:
csv_raw.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
mes_id,16172.0,NaN,NaN,NaN,201362.137336,769.772278,200001.0,200707.0,201405.0,202008.0,202609.0
aeropuerto_oaci,16172,69,SCAR,642,NaN,NaN,NaN,NaN,NaN,NaN,NaN
internacional_domestico,16172,2,D,12138,NaN,NaN,NaN,NaN,NaN,NaN,NaN
cnt_operaciones,16172.0,NaN,NaN,NaN,660.540069,1211.690644,1.0,23.0,172.0,782.0,10722.0


In [12]:
csv_raw.nunique()

mes_id                      321
aeropuerto_oaci              69
internacional_domestico       2
cnt_operaciones            2837
dtype: int64

### 5.2 Bitácora de vuelos (Parquet)

In [13]:
bit_raw.head()

,dt_operacion,aeropuerto_oaci,tipo_operacion,aeropuerto_dgac_orig_dest,aerolinea_dgac,numero_vuelo,actividad_cod,matricula,modelo_avion,modelo_avion_desc,es_internacional,pmd
0,1999-12-31 21:00:00-03:00,SCEL,A,SPIM,LAN,631,U,CCCDM,B763,767-352ER,True,187.00
1,1999-12-31 21:00:00-03:00,SCTC,D,SCUG,6,0,P,CCSPC,C172,172B,False,1.00
2,1999-12-31 21:04:00.001000-03:00,SCEL,A,SCTC,LAN,4248,U,CCCHS,EC35,EC-135 T2,False,2.84
3,1999-12-31 21:08:00-03:00,SCFA,A,SCEL,LCO,2395,U,CCCYK,B732,737-205,False,53.00
4,1999-12-31 21:08:00-03:00,SCFA,A,SCEL,LCO,2394,U,CCCYK,B732,737-205,False,53.00


In [14]:
bit_raw.tail()

,dt_operacion,aeropuerto_oaci,tipo_operacion,aeropuerto_dgac_orig_dest,aerolinea_dgac,numero_vuelo,actividad_cod,matricula,modelo_avion,modelo_avion_desc,es_internacional,pmd
11163786,2026-09-30 20:53:00-03:00,SCSE,D,SCCF,JAT,136,U,CCDIN,A320,A320-271N,False,77.0
11163787,2026-09-30 20:54:00-03:00,SCCF,D,SCEL,LAN,1859,U,CCBEO,A321,A-321-211,False,89.0
11163788,2026-09-30 20:55:00-03:00,SCEL,A,SCQP,LAN,34,U,CCBEI,A321,A-321-211,False,89.0
11163789,2026-09-30 20:57:00-03:00,SCIE,D,SCEL,LAN,10,U,CCBEL,A321,A-321-211,False,89.0
11163790,2026-09-30 20:58:00-03:00,SCCF,A,SCSE,JAT,134,U,CCAWB,A320,A-320-232,False,78.0


In [15]:
bit_raw.sample(5, random_state=42)

,dt_operacion,aeropuerto_oaci,tipo_operacion,aeropuerto_dgac_orig_dest,aerolinea_dgac,numero_vuelo,actividad_cod,matricula,modelo_avion,modelo_avion_desc,es_internacional,pmd
10816381,2026-02-07 23:40:00-03:00,SCEL,A,SCAR,SKU,307,U,CCDBL,A320,A320-251N,False,79.00
1338062,2003-12-30 15:59:00-03:00,SCTB,D,SCTB,6,0,P,CCKIJ,C152,152,False,0.76
2619068,2007-10-14 12:32:00-03:00,SCFA,D,SCFA,6,,P,CCNAC,P28A,PA-28-180 CHERO,False,1.09
596927,2001-11-11 20:23:00-03:00,SCTI,A,SCTB,3R,1,O,CCCRZ,PA38,PA-38-112,False,0.76
5873112,2015-12-08 14:11:00-03:00,SCJO,D,SCJO,21R,0,O,CCCXS,C206,206,False,1.73


In [16]:
print("shape:", bit_raw.shape)
bit_raw.info(memory_usage="deep")

shape: (11163791, 12)
<class 'pandas.DataFrame'>
RangeIndex: 11163791 entries, 0 to 11163790
Data columns (total 12 columns):
 #   Column                     Dtype                           
---  ------                     -----                           
 0   dt_operacion               datetime64[us, America/Santiago]
 1   aeropuerto_oaci            category                        
 2   tipo_operacion             category                        
 3   aeropuerto_dgac_orig_dest  category                        
 4   aerolinea_dgac             category                        
 5   numero_vuelo               category                        
 6   actividad_cod              category                        
 7   matricula                  category                        
 8   modelo_avion               category                        
 9   modelo_avion_desc          category                        
 10  es_internacional           bool                            
 11  pmd                     

In [17]:
bit_raw.describe(include="all").T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
dt_operacion,11163791,NaN,NaN,NaN,2014-09-10 17:16:20.947549-03:00,1999-12-31 21:00:00-03:00,2008-03-27 19:51:00.000500-03:00,2015-03-28 18:57:00.001000-03:00,2021-09-05 14:33:00.000500-03:00,2026-09-30 20:58:00-03:00,NaN
aeropuerto_oaci,11163791,69,SCEL,3146276,NaN,NaN,NaN,NaN,NaN,NaN,NaN
tipo_operacion,11163791,4,A,5356831,NaN,NaN,NaN,NaN,NaN,NaN,NaN
aeropuerto_dgac_orig_dest,11163791,1671,SCEL,1713457,NaN,NaN,NaN,NaN,NaN,NaN,NaN
aerolinea_dgac,11163791,2568,LAN,1613920,NaN,NaN,NaN,NaN,NaN,NaN,NaN
numero_vuelo,10681646,7442,,1698594,NaN,NaN,NaN,NaN,NaN,NaN,NaN
actividad_cod,11163791,26,U,6157517,NaN,NaN,NaN,NaN,NaN,NaN,NaN
matricula,11163791,13835,CCBAY,48564,NaN,NaN,NaN,NaN,NaN,NaN,NaN
modelo_avion,11163791,1286,A320,1769307,NaN,NaN,NaN,NaN,NaN,NaN,NaN
modelo_avion_desc,11163791,2325,A-321-211,595007,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
bit_raw.nunique().sort_values()

es_internacional                   2
tipo_operacion                     4
actividad_cod                     26
aeropuerto_oaci                   69
pmd                              785
modelo_avion                    1286
aeropuerto_dgac_orig_dest       1671
modelo_avion_desc               2325
aerolinea_dgac                  2568
numero_vuelo                    7442
matricula                      13835
dt_operacion                 6527475
dtype: int64

### 5.3 Perfil consolidado

In [19]:
def perfil(df: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "cardinalidad": df.nunique(),
        "nulos": df.isna().sum(),
        "%_nulos": (df.isna().mean() * 100).round(2),
    })
    vacios = {}
    for c in df.columns:
        if isinstance(df[c].dtype, pd.CategoricalDtype) or df[c].dtype == object or str(df[c].dtype) == "str":
            s = df[c].astype("string")
            vacios[c] = int((s.str.strip() == "").sum())
    out["cadenas_vacías"] = pd.Series(vacios)
    return out


print("CSV — registros:", len(csv_raw), "| columnas:", csv_raw.shape[1],
      "| duplicados exactos:", csv_raw.duplicated().sum())
display(perfil(csv_raw))
print("Bitácora — registros:", f"{len(bit_raw):,}", "| columnas:", bit_raw.shape[1],
      "| duplicados exactos:", f"{bit_raw.duplicated().sum():,}")
display(perfil(bit_raw))

CSV — registros: 16172 | columnas: 4 | duplicados exactos: 0


,dtype,cardinalidad,nulos,%_nulos,cadenas_vacías
mes_id,int64,321,0,0.0,NaN
aeropuerto_oaci,str,69,0,0.0,0.0
internacional_domestico,str,2,0,0.0,0.0
cnt_operaciones,int64,2837,0,0.0,NaN


Bitácora — registros: 11,163,791 | columnas: 12 | duplicados exactos: 1,122


,dtype,cardinalidad,nulos,%_nulos,cadenas_vacías
dt_operacion,"datetime64[us, America/Santiago]",6527475,0,0.00,NaN
aeropuerto_oaci,category,69,0,0.00,0.0
tipo_operacion,category,4,0,0.00,0.0
aeropuerto_dgac_orig_dest,category,1671,0,0.00,0.0
aerolinea_dgac,category,2568,0,0.00,0.0
numero_vuelo,category,7442,482145,4.32,2865981.0
actividad_cod,category,26,0,0.00,0.0
matricula,category,13835,0,0.00,0.0
modelo_avion,category,1286,0,0.00,6.0
modelo_avion_desc,category,2325,0,0.00,36763.0


**Clasificación inicial de variables**

* **Numéricas:** `cnt_operaciones` (discreta, CSV), `pmd` (continua, toneladas).
* **Categóricas:** `aeropuerto_oaci`, `internacional_domestico`, `tipo_operacion`, `aerolinea_dgac`,
  `actividad_cod`, `modelo_avion`, `aeropuerto_dgac_orig_dest`; `es_internacional` (booleana).
* **Temporales:** `mes_id` (entero AAAAMM, CSV), `dt_operacion` (fecha-hora con zona, bitácora).
* **Identificadores / alta cardinalidad:** `matricula` (13.835), `numero_vuelo`, `modelo_avion_desc`.
* **Espaciales:** no hay coordenadas; sólo códigos OACI de aeropuerto.

## 6. Auditoría de calidad (Partes IX y X)

Se investiga mediante código cada problema potencial. El objetivo es **detectar lo que
realmente existe**, no inventar problemas.

In [20]:
print("Nulos CSV:\n", csv_raw.isna().sum(), sep="")
print("\nDuplicados exactos CSV:", csv_raw.duplicated().sum())
print("Duplicados de clave (mes, aeropuerto, tipo):",
      csv_raw.duplicated(["mes_id", "aeropuerto_oaci", "internacional_domestico"]).sum())
print("Valores de internacional_domestico:", csv_raw["internacional_domestico"].unique())
print("cnt_operaciones <= 0:", (csv_raw["cnt_operaciones"] <= 0).sum())
meses = csv_raw["mes_id"] % 100
print("Meses fuera de 1-12:", (~meses.between(1, 12)).sum())
print("Rango mes_id:", csv_raw["mes_id"].min(), "→", csv_raw["mes_id"].max(),
      "| meses distintos:", csv_raw["mes_id"].nunique(),
      "| meses esperados ene-2000 a sep-2026:", 26 * 12 + 9)

Nulos CSV:
mes_id                     0
aeropuerto_oaci            0
internacional_domestico    0
cnt_operaciones            0
dtype: int64

Duplicados exactos CSV: 0
Duplicados de clave (mes, aeropuerto, tipo): 0
Valores de internacional_domestico: <ArrowStringArray>
['D', 'I']
Length: 2, dtype: str
cnt_operaciones <= 0: 0
Meses fuera de 1-12: 0
Rango mes_id: 200001 → 202609 | meses distintos: 321 | meses esperados ene-2000 a sep-2026: 321


In [21]:
cobertura = (csv_raw.groupby("aeropuerto_oaci")
             .agg(meses_con_datos=("mes_id", "nunique"), primer_mes=("mes_id", "min"),
                  ultimo_mes=("mes_id", "max"), total_operaciones=("cnt_operaciones", "sum"))
             .sort_values("total_operaciones", ascending=False))
print("Aeropuertos con serie completa (321 meses):", (cobertura["meses_con_datos"] == 321).sum(),
      "de", len(cobertura))
cobertura.head(12)

Aeropuertos con serie completa (321 meses): 17 de 69


,meses_con_datos,primer_mes,ultimo_mes,total_operaciones
aeropuerto_oaci,,,,
SCEL,321,200001,202609,3132971
SCTB,321,200001,202609,1284352
SCIE,321,200001,202609,526012
SCFA,321,200001,202609,491290
SCDA,321,200001,202609,440760
SCTE,321,200001,202609,423980
SCSE,321,200001,202609,316396
SCPF,219,200807,202609,309651
SCRD,321,200001,202609,296260


In [22]:
# Aeropuertos que dejan de informar o aparecen tarde: cambios reales de infraestructura
cobertura.loc[["SCTI", "SCTC", "SCQP", "SCHA", "SCAT", "SCLC", "SCPF", "SCRG"]]

,meses_con_datos,primer_mes,ultimo_mes,total_operaciones
aeropuerto_oaci,,,,
SCTI,69,200001,200601,213350
SCTC,175,200001,201407,167584
SCQP,147,201407,202609,107272
SCHA,63,200001,200901,23495
SCAT,260,200502,202609,127788
SCLC,177,200001,202605,148105
SCPF,219,200807,202609,309651
SCRG,91,201903,202609,110967


In [23]:
# Concentración y asimetría de la medida
print("Asimetría (skew) de cnt_operaciones:", round(csv_raw["cnt_operaciones"].skew(), 2))
print("Registros con 1 sola operación:", (csv_raw["cnt_operaciones"] == 1).sum())
print("Participación de los 2 mayores aeropuertos en el total:",
      f"{cobertura['total_operaciones'].head(2).sum() / cobertura['total_operaciones'].sum():.1%}")

Asimetría (skew) de cnt_operaciones: 3.49
Registros con 1 sola operación: 425
Participación de los 2 mayores aeropuertos en el total: 41.4%


### 6.2 Bitácora

In [24]:
print("Nulos:\n", bit_raw.isna().sum(), sep="")
print("\nValores de tipo_operacion:", bit_raw["tipo_operacion"].value_counts().to_dict())
bit_raw[bit_raw["tipo_operacion"] == "t"]

Nulos:
dt_operacion                      0
aeropuerto_oaci                   0
tipo_operacion                    0
aeropuerto_dgac_orig_dest         0
aerolinea_dgac                    0
numero_vuelo                 482145
actividad_cod                     0
matricula                         0
modelo_avion                      0
modelo_avion_desc                 0
es_internacional                  0
pmd                          275115
dtype: int64



Valores de tipo_operacion: {'A': 5356831, 'D': 5325353, 'W': 481606, 't': 1}


,dt_operacion,aeropuerto_oaci,tipo_operacion,aeropuerto_dgac_orig_dest,aerolinea_dgac,numero_vuelo,actividad_cod,matricula,modelo_avion,modelo_avion_desc,es_internacional,pmd
6375752,2017-01-16 17:23:00-03:00,SCSN,t,SCSN,6I,1,O,CCAFD,C172,172S,False,1.16


In [25]:
for c in ["numero_vuelo", "modelo_avion", "modelo_avion_desc"]:
    s = bit_raw[c].astype("string")
    print(f"{c:18s} nulos={s.isna().sum():>9,}  cadenas vacías={(s.str.strip() == '').sum():>9,}")

numero_vuelo       nulos=  482,145  cadenas vacías=2,865,981


modelo_avion       nulos=        0  cadenas vacías=        6


modelo_avion_desc  nulos=        0  cadenas vacías=   36,763


In [26]:
dups = bit_raw[bit_raw.duplicated(keep=False)]
print(f"Filas duplicadas exactas (sobran): {bit_raw.duplicated().sum():,}")
print("Por aeropuerto:", dups["aeropuerto_oaci"].value_counts().head(4).to_dict())
print("Por año:", dups["dt_operacion"].dt.year.value_counts().head(4).to_dict())
dups.sort_values(["matricula", "dt_operacion"]).head(6)

Filas duplicadas exactas (sobran): 1,122
Por aeropuerto: {'SCVM': 1285, 'SCTI': 50, 'SCTB': 46, 'SCTC': 41}
Por año: {2007: 594, 2004: 433, 2005: 340, 2002: 71}


,dt_operacion,aeropuerto_oaci,tipo_operacion,aeropuerto_dgac_orig_dest,aerolinea_dgac,numero_vuelo,actividad_cod,matricula,modelo_avion,modelo_avion_desc,es_internacional,pmd
15,1999-12-31 21:55:00-03:00,SCFA,D,SCDA,LCO,2394,U,CCCYK,B732,737-205,False,53.0
16,1999-12-31 21:55:00-03:00,SCFA,D,SCDA,LCO,2394,U,CCCYK,B732,737-205,False,53.0
517,2000-01-01 21:31:00-03:00,SCFA,A,SCEL,LCO,394,U,CCCYK,B732,737-205,False,53.0
518,2000-01-01 21:31:00-03:00,SCFA,A,SCEL,LCO,394,U,CCCYK,B732,737-205,False,53.0
10,1999-12-31 21:39:00-03:00,SCFA,A,SCEL,LAN,158,U,CCCJW,B732,737-200,False,53.0
11,1999-12-31 21:39:00-03:00,SCFA,A,SCEL,LAN,158,U,CCCJW,B732,737-200,False,53.0


In [27]:
pmd = bit_raw["pmd"]
print(pmd.describe().round(2))
print("\npmd == 0:", (pmd == 0).sum(), "| pmd nulo:", pmd.isna().sum(),
      f"| pmd > {prep.PMD_MAX_PLAUSIBLE_T:.0f} t:", (pmd > prep.PMD_MAX_PLAUSIBLE_T).sum())
(bit_raw[pmd > prep.PMD_MAX_PLAUSIBLE_T]
 .groupby("modelo_avion", observed=True)["pmd"].agg(["count", "min", "max"])
 .sort_values("count", ascending=False))

count    10888676.00
mean           45.08
std            61.35
min             0.00
25%             1.16
50%             7.00
75%            77.00
max          2100.00
Name: pmd, dtype: float64

pmd == 0: 25823 | pmd nulo: 275115 | pmd > 600 t: 1112


,count,min,max
modelo_avion,,,
T34T,1007,1340.0,1340.0
A225,34,640.0,640.0
AL3,28,2100.0,2100.0
C210,20,1500.0,1500.0
R44,15,1000.0,1000.0
B38M,4,710.0,710.0
C206,2,1640.0,1640.0
F33A,2,1450.0,1450.0


In [28]:
# Referencia: el PMD de un mismo modelo es estable (p. ej. A320 ~ 73-79 t, C172 ~ 1,1 t)
for m in ["A320", "C172", "T34T"]:
    print(m, bit_raw.loc[bit_raw["modelo_avion"] == m, "pmd"].value_counts().head(3).to_dict())

A320 {77.0: 1278283, 79.0: 218014, 78.0: 127058}
C172 {1.04: 511047, 1.1600000000000001: 273976, 1.09: 87946}
T34T {1340.0: 1007}


In [29]:
print("Rango dt_operacion:", bit_raw["dt_operacion"].min(), "→", bit_raw["dt_operacion"].max())
print("Zona horaria almacenada:", bit_raw["dt_operacion"].dt.tz)
print("Registros con fecha local en 1999:", (bit_raw["dt_operacion"].dt.year == 1999).sum())
print("Código de aerolínea numérico (aviación general / privados):",
      bit_raw["aerolinea_dgac"].astype(str).str.match(r"^\d").mean().round(3))

Rango dt_operacion: 1999-12-31 21:00:00-03:00 → 2026-09-30 20:58:00-03:00
Zona horaria almacenada: America/Santiago


Registros con fecha local en 1999: 41


Código de aerolínea numérico (aviación general / privados): 0.428
